# Fintech Sandbox 기업소개 상세 크롤러 (v5)
716건 수집을 목표로 목록/상세 양쪽을 강화하고, 수집 완료 후 **DataFrame 특수문자 정제**까지 수행합니다.

In [ ]:
# %pip install -U selenium webdriver-manager pandas lxml

In [3]:

# -*- coding: utf-8 -*-
"""
Fintech Sandbox '기업소개' 상세 크롤러 (v5 - 최대수집 + 정제)
(설명은 이전 셀과 동일)
"""
import re, csv, time, random
from typing import List, Dict, Optional, Tuple
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

BASE_LIST_URL = "https://sandbox.fintech.or.kr/business/enterprise_intro.do?pageIndex={page}"
DETAIL_BASE   = "https://sandbox.fintech.or.kr/business/enterprise.do?lang=ko&id={id}"

MAX_PAGE        = 72
HEADLESS        = True
TIMEOUT_SEC     = 14
RETRY_DETAIL    = 2
SLOW_MIN, SLOW_MAX = 0.35, 0.9
OUT_RAW_CSV     = f"./fintech_sandbox_v5_raw_{int(time.time())}.csv"
OUT_CLEAN_CSV   = f"./fintech_sandbox_v5_clean_{int(time.time())}.csv"
EMPTY_VALUE     = ""
TARGET_KEYS = ["서비스명","지정 제도","서비스 주요 내용","규제 특례 내용","주요 부가 조건 내용"]

def build_driver(headless=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1400,2600")
    opts.add_argument("--lang=ko-KR")
    opts.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    drv = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)
    drv.set_page_load_timeout(50)
    return drv

def _remove_artifact_tokens(s: str) -> str:
    if not s:
        return s
    s = re.sub(r"</?LAW[^>]*>", "", s, flags=re.IGNORECASE)
    s = re.sub(r"<\s*LAW[_A-Za-z0-9\-]*\s*>", "", s, flags=re.IGNORECASE)
    return s

import re
BULLET_START = re.compile(r"^(\s*(?:[\-–—•·/]|[①-⑳]|\(?\d{1,3}\)?[.)]|[가-힣]\))\s*)")

def _strip_leading_bullets_once(line: str) -> str:
    m = BULLET_START.match(line)
    return line[m.end():] if m else line

def _strip_leading_bullets_multiline(text: str) -> str:
    out_lines = []
    for ln in text.splitlines():
        s = ln
        if re.match(r"^\s*제\s*\d+\s*조", s):
            out_lines.append(s.strip())
            continue
        prev = None
        while prev != s:
            prev = s
            s = _strip_leading_bullets_once(s)
        out_lines.append(s.strip())
    return "\n".join(out_lines).strip()

def clean_text_keep_law(s: str) -> str:
    if not s:
        return ""
    s = s.replace("\r","\n").replace("\xa0"," ").strip()
    s = re.sub(r"\n{3,}", "\n\n", s)
    s = "\n".join(line.strip() for line in s.splitlines())
    s = re.sub(r"[ \t]{2,}", " ", s)
    s = _remove_artifact_tokens(s)
    s = _strip_leading_bullets_multiline(s)
    return s

def find_main_list_table(drv):
    tables = drv.find_elements(By.TAG_NAME, "table")
    best, best_score = None, -1
    header_tokens = ["No", "NO", "no", "기업", "회사", "서비스", "상세", "보기", "등록일"]
    for t in tables:
        try:
            ths = t.find_elements(By.CSS_SELECTOR, "thead th")
            text = ""
            if ths:
                text = " ".join(clean_text_keep_law(th.get_attribute("innerText")) for th in ths)
            else:
                first_tr = t.find_elements(By.CSS_SELECTOR, "tr")
                if first_tr:
                    tds = first_tr[0].find_elements(By.CSS_SELECTOR, "th,td")
                    text = " ".join(clean_text_keep_law(x.get_attribute("innerText")) for x in tds)
            score = sum(1 for tok in header_tokens if tok in text)
            rows = t.find_elements(By.CSS_SELECTOR, "tbody tr")
            if len(rows) >= 5:
                score += 1
            if score > best_score:
                best, best_score = t, score
        except StaleElementReferenceException:
            continue
    return best

def extract_detail_url_from_row(row) -> Optional[str]:
    cands = []
    try:
        cands.append(row)
        cands.extend(row.find_elements(By.XPATH, ".//td[5]"))
        cands.extend(row.find_elements(By.XPATH, ".//td[5]//*"))
        cands.extend(row.find_elements(By.XPATH, ".//a|.//button|.//em"))
    except Exception:
        pass
    for el in cands:
        try:
            href = (el.get_attribute("href") or "").strip()
            onclick = (el.get_attribute("onclick") or "").strip()
            if "enterprise.do" in href:
                return href
            m = re.search(r"id=(\d{1,10})", onclick)
            if m:
                return DETAIL_BASE.format(id=m.group(1))
            m2 = re.search(r"['\"]?(\d{1,10})['\"]?\)", onclick)
            if m2:
                return DETAIL_BASE.format(id=m2.group(1))
        except StaleElementReferenceException:
            continue
    return None

def extract_detail_urls_on_list_page(drv) -> List[str]:
    urls: List[str] = []
    table = find_main_list_table(drv)
    if table:
        rows = table.find_elements(By.CSS_SELECTOR, "tbody tr")
        for i, row in enumerate(rows, 1):
            url = extract_detail_url_from_row(row)
            if url:
                urls.append(url)
            else:
                print(f"    [MISS row {i}] 상세 링크 없음")
    html = drv.page_source
    urls += re.findall(r"https?://sandbox\.fintech\.or\.kr/business/enterprise\.do\?lang=ko&id=\d+", html)
    for m in re.findall(r"goView\((?:'|\"|)(\d{1,10})(?:'|\"|)\)", html):
        urls.append(DETAIL_BASE.format(id=m))
    seen = set(); uniq = []
    for u in urls:
        if u not in seen:
            seen.add(u); uniq.append(u)
    return uniq

def find_section_table(drv):
    anchors = drv.find_elements(By.XPATH, "//*[contains(normalize-space(.),'샌드박스 지정 내용 및 성과')]")
    if anchors:
        try:
            tbl = drv.find_element(By.XPATH, "(//*[contains(normalize-space(.),'샌드박스 지정 내용 및 성과')])[1]/following::table[1]")
            return tbl
        except NoSuchElementException:
            pass
    try:
        return drv.find_element(By.XPATH, "/html/body/div/div[2]/div[3]/div[2]/div[7]/div[2]/table")
    except NoSuchElementException:
        pass
    try:
        tables = drv.find_elements(By.TAG_NAME, "table")
    except Exception:
        return None
    best, best_score = None, -1
    keys = ["서비스", "지정", "규제", "부가", "내용"]
    for t in tables:
        try:
            text = t.get_attribute("innerText") or ""
            score = sum(1 for k in keys if k in text)
            if score > best_score and len(text) > 30:
                best, best_score = t, score
        except StaleElementReferenceException:
            continue
    return best

def parse_table_to_map(tbl) -> Dict[str,str]:
    data = {}
    rows = tbl.find_elements(By.CSS_SELECTOR, "tr")
    for r in rows:
        label = ""
        try:
            ths = r.find_elements(By.CSS_SELECTOR, "th")
            if ths:
                label = clean_text_keep_law(ths[0].get_attribute("innerText")).replace(" ","")
            else:
                tds = r.find_elements(By.CSS_SELECTOR, "td")
                if tds:
                    maybe = clean_text_keep_law(tds[0].get_attribute("innerText"))
                    if len(maybe) <= 12:
                        label = maybe.replace(" ","")
        except Exception:
            pass
        value = ""
        try:
            tds = r.find_elements(By.CSS_SELECTOR, "td")
            if tds:
                value = clean_text_keep_law(tds[-1].get_attribute("innerText"))
        except Exception:
            pass
        if label:
            data[label] = value
    return data

def normalize_regulatory_value(val: str) -> str:
    v = (val or "").strip()
    if v == "" or v in ["-", "없음", "해당없음", "해당 없음", "미해당"]:
        return EMPTY_VALUE
    if re.fullmatch(r"(없음|해당없음|해당 없음|미해당)[\.\s]*", v or ""):
        return EMPTY_VALUE
    return v

def extract_service_name_fallbacks(drv) -> str:
    for xp in [
        "//h3[contains(.,'서비스')]/following::*[self::p or self::div][1]",
        "//h2|//h3|//h4"
    ]:
        try:
            el = drv.find_element(By.XPATH, xp)
            t = clean_text_keep_law(el.get_attribute("innerText"))
            if t and len(t) <= 120 and "서비스" in t:
                return t
        except Exception:
            pass
    try:
        el = drv.find_element(By.XPATH, "//*[contains(normalize-space(.),'서비스명')]/following::*[1]")
        return clean_text_keep_law(el.get_attribute("innerText"))
    except Exception:
        return ""

def extract_fields_from_detail(drv) -> Dict[str,str]:
    result = {k:"" for k in TARGET_KEYS}
    tbl = find_section_table(drv)
    if tbl:
        m = parse_table_to_map(tbl)
        alias = {
            "지정제도":"지정 제도",
            "지정제도구분":"지정 제도",
            "지정 제도":"지정 제도",
            "서비스명":"서비스명",
            "서비스 명":"서비스명",
            "서비스주요내용":"서비스 주요 내용",
            "서비스 주요 내용":"서비스 주요 내용",
            "규제특례내용":"규제 특례 내용",
            "규제 특례 내용":"규제 특례 내용",
            "주요부가조건내용":"주요 부가 조건 내용",
            "주요 부가 조건 내용":"주요 부가 조건 내용",
        }
        for k_src, v in m.items():
            k_norm = alias.get(k_src)
            if k_norm in result:
                result[k_norm] = v
        def td_by_row(idx: int) -> str:
            try:
                el = tbl.find_element(By.XPATH, f".//tbody/tr[{idx}]/td")
                return clean_text_keep_law(el.get_attribute("innerText"))
            except Exception:
                return ""
        if not result["지정 제도"]:
            result["지정 제도"] = td_by_row(1)
        if not result["서비스명"]:
            result["서비스명"] = td_by_row(2)
        if not result["서비스 주요 내용"]:
            result["서비스 주요 내용"] = td_by_row(3)
        if not result["규제 특례 내용"]:
            result["규제 특례 내용"] = td_by_row(5)
        if not result["주요 부가 조건 내용"]:
            result["주요 부가 조건 내용"] = td_by_row(6)

    if not result["서비스명"]:
        result["서비스명"] = extract_service_name_fallbacks(drv)

    result["규제 특례 내용"] = normalize_regulatory_value(result.get("규제 특례 내용",""))
    return result

# ---- DF 특수문자 정제 ----
ZW_CHARS = "[\\u200B-\\u200F\\u202A-\\u202E\\u2060-\\u206F\\uFEFF]"
BULLETS = "•·●○◆◇■□▶▷◀◁▲△▼▽※☆★→←↔⇒⇔◦ㆍ❖➤➔➜➝✓✔✗✘"

def sanitize_text(s: str) -> str:
    if not isinstance(s, str):
        return s
    t = s
    t = re.sub(ZW_CHARS, "", t)
    t = t.translate({ord(ch): None for ch in BULLETS})
    t = re.sub(r"^[\-\=_]{2,}$", "", t, flags=re.MULTILINE)
    t = re.sub(r"</?LAW[^>]*>", "", t, flags=re.IGNORECASE)
    t = re.sub(r"[ \t]{2,}", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t).strip()
    return t

def sanitize_df(df: pd.DataFrame) -> pd.DataFrame:
    clean_df = df.copy()
    for col in clean_df.columns:
        clean_df[col] = clean_df[col].map(sanitize_text)
    return clean_df

def crawl(max_page: int = MAX_PAGE,
          out_raw_csv: str = OUT_RAW_CSV,
          out_clean_csv: str = OUT_CLEAN_CSV,
          headless: bool = HEADLESS):
    drv = build_driver(headless=headless)
    collected = []
    bad_links = []
    try:
        for page in range(1, max_page+1):
            list_url = BASE_LIST_URL.format(page=page)
            drv.get(list_url)
            try:
                WebDriverWait(drv, TIMEOUT_SEC).until(EC.presence_of_element_located((By.XPATH, "//table//tbody//tr")))
            except TimeoutException:
                print(f"[WARN] 목록 테이블 로드 타임아웃 - page {page}")
                continue
            time.sleep(random.uniform(SLOW_MIN, SLOW_MAX))

            detail_urls = extract_detail_urls_on_list_page(drv)
            print(f"[PAGE {page}] 상세 {len(detail_urls)}건")
            for i, du in enumerate(detail_urls, 1):
                ok = False
                for attempt in range(1, RETRY_DETAIL+2):
                    try:
                        drv.get(du)
                        WebDriverWait(drv, TIMEOUT_SEC).until(EC.presence_of_element_located((By.CSS_SELECTOR, "body")))
                        time.sleep(random.uniform(0.2, 0.6))
                        item = extract_fields_from_detail(drv)
                        row = {k: item.get(k,"") for k in TARGET_KEYS}
                        collected.append(row)
                        print(f"  - ({i}/{len(detail_urls)}) OK: {row.get('서비스명','')[:80]}")
                        ok = True
                        break
                    except TimeoutException:
                        print(f"  - ({i}/{len(detail_urls)}) TIMEOUT[{attempt}]: {du}")
                        time.sleep(0.5 + 0.2*attempt)
                    except WebDriverException as e:
                        print(f"  - ({i}/{len(detail_urls)}) WEBDRV[{attempt}] ERR: {e}")
                        time.sleep(0.5 + 0.2*attempt)
                    except Exception as e:
                        print(f"  - ({i}/{len(detail_urls)}) ERR[{attempt}]: {e}")
                        time.sleep(0.5 + 0.2*attempt)
                if not ok:
                    bad_links.append(du)

            time.sleep(random.uniform(SLOW_MIN, SLOW_MAX))

        df = pd.DataFrame(collected, columns=TARGET_KEYS)
        df.to_csv(out_raw_csv, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
        print(f"Saved RAW CSV -> {out_raw_csv} (rows={len(df)})")

        df_clean = sanitize_df(df)
        df_clean.to_csv(out_clean_csv, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
        print(f"Saved CLEAN CSV -> {out_clean_csv} (rows={len(df_clean)})")
        if bad_links:
            print(f"[INFO] 재시도 후 실패 링크 {len(bad_links)}건")
        return df, df_clean
    finally:
        try:
            drv.quit()
        except Exception:
            pass

if __name__ == "__main__":
    raw_df, clean_df = crawl()


[PAGE 1] 상세 10건
  - (1/10) OK: 생성형 AI를 활용한 가입설계 AI Agent 서비스
  - (2/10) OK: 외국인 보험 설계사를 위한 생성형AI 번역 서비스
  - (3/10) OK: Persuit
  - (4/10) OK: Passport
  - (5/10) OK: Talent Marketplace
  - (6/10) OK: Talent Marketplace
  - (7/10) OK: T-PACE
  - (8/10) OK: AI 기반 지능형 고객채팅문의 자동화 서비스 (Finda AI Chat Assistant)
  - (9/10) OK: 생성형 AI를 이용한 투자정보 분석 서비스
  - (10/10) OK: 생성형 AI를 이용한 챗봇 서비스
[PAGE 2] 상세 10건
  - (1/10) OK: 클라우드를 활용한 분석 및 자동화 솔루션 도입
  - (2/10) OK: 클라우드를 활용한 구매 관리 소프트웨어 서비스(SaaS)의 내부망 이용(Levelpath)
  - (3/10) OK: 클라우드를 활용한 구매 관리 소프트웨어 서비스(SaaS)의 내부망 이용(Levelpath)
  - (4/10) OK: 생성형 AI를 활용한 업무생산성 향상 챗봇 서비스
  - (5/10) OK: AI 기반 상담 Assistant 서비스
  - (6/10) OK: 생성형 AI를 활용한 앱 정보 번역 서비스
  - (7/10) OK: 생성형AI 기반 마케팅 업무 자동화 및 효율화
  - (8/10) OK: 생성형AI를 활용한 금융 맞춤형 합성 데이터 생성 및 AI모델 평가 서비스
  - (9/10) OK: 페이아이
  - (10/10) OK: SaaS기반의 클라우드 서비스(Slack) 내부망 활용을 통한 업무 생산성 향상
[PAGE 3] 상세 10건
  - (1/10) OK: 클라우드를 활용한 협업툴 소프트웨어 서비스 ‘플로우(Flow)’의 내부망 이용
  - (2/10) OK: 클라우드(SaaS)를 활용한 협업툴 소프트웨어 서비스(ServiceNow)

In [4]:
raw_df, clean_df = crawl()
print('RAW shape:', raw_df.shape)
print('CLEAN shape:', clean_df.shape)
clean_df.head()

[PAGE 1] 상세 10건
  - (1/10) OK: 생성형 AI를 활용한 가입설계 AI Agent 서비스
  - (2/10) OK: 외국인 보험 설계사를 위한 생성형AI 번역 서비스
  - (3/10) OK: Persuit
  - (4/10) OK: Passport
  - (5/10) OK: Talent Marketplace
  - (6/10) OK: Talent Marketplace
  - (7/10) OK: T-PACE
  - (8/10) OK: AI 기반 지능형 고객채팅문의 자동화 서비스 (Finda AI Chat Assistant)
  - (9/10) OK: 생성형 AI를 이용한 투자정보 분석 서비스
  - (10/10) OK: 생성형 AI를 이용한 챗봇 서비스
[PAGE 2] 상세 10건
  - (1/10) OK: 클라우드를 활용한 분석 및 자동화 솔루션 도입
  - (2/10) OK: 클라우드를 활용한 구매 관리 소프트웨어 서비스(SaaS)의 내부망 이용(Levelpath)
  - (3/10) OK: 클라우드를 활용한 구매 관리 소프트웨어 서비스(SaaS)의 내부망 이용(Levelpath)
  - (4/10) OK: 생성형 AI를 활용한 업무생산성 향상 챗봇 서비스
  - (5/10) OK: AI 기반 상담 Assistant 서비스
  - (6/10) OK: 생성형 AI를 활용한 앱 정보 번역 서비스
  - (7/10) OK: 생성형AI 기반 마케팅 업무 자동화 및 효율화
  - (8/10) OK: 생성형AI를 활용한 금융 맞춤형 합성 데이터 생성 및 AI모델 평가 서비스
  - (9/10) OK: 페이아이
  - (10/10) OK: SaaS기반의 클라우드 서비스(Slack) 내부망 활용을 통한 업무 생산성 향상
[PAGE 3] 상세 10건
  - (1/10) OK: 클라우드를 활용한 협업툴 소프트웨어 서비스 ‘플로우(Flow)’의 내부망 이용
  - (2/10) OK: 클라우드(SaaS)를 활용한 협업툴 소프트웨어 서비스(ServiceNow)

,서비스명,지정 제도,서비스 주요 내용,규제 특례 내용,주요 부가 조건 내용
0,생성형 AI를 활용한 가입설계 AI Agent 서비스,혁신금융서비스,"생성형 AI가 보험설계사의 간단한 요청만으로 가입설계 전반의 과정을 대행하여, 보험...",전자금융감독규정 제15조 제1항 제5호,규제 특례사항에 대한 보안위협에 대비한 보안요건을 준수할 것\n신청자가 기 수립한 ...
1,외국인 보험 설계사를 위한 생성형AI 번역 서비스,혁신금융서비스,"소속 보험설계사 대상으로, MS사가 제공하는 생성형 AI(MS Azure OpenA...",전자금융감독원규정 제15조 제1항 제5호,규제 특례사항에 대해 보안위혐에 대비한 보안 요건을 준수할 것\n신청자가 기 수립한...
2,Persuit,혁신금융서비스,임직원의 업무 생산성 향상을 위해 Persuit (Australia) Operati...,전자금융 감독규정 제15조 (해킹 등 방지대책) 제1항 제3호 에 대한 특례,규제 특례사항에 대해 보안위협에 대비한 보안 요건을 준수할 것\n신청자가 기 수립한...
3,Passport,혁신금융서비스,임직원의 업무 생산성 향상을 위해 볼터스클루버社가 제공하는 SaaS(Software...,전자금융 감독규정 제15조 (해킹 등 방지대책) 제1항 제3호 에 대한 특례,규제 특례사항에 대해 보안위협에 대비한 보안 요건을 준수할 것\n신청자가 기 수립한...
4,Talent Marketplace,혁신금융서비스,"Talent Marketplace는 AI 기반의 내부인재 관리 플랫폼으로, 직원들의...",전자금융감독규정 제15조 (해킹 등 방지대책) 제1항 제3호,규제 특례사항에 대해 보안위협에 대비한 보안 요건 준수\n혁신금용서비스 지정시 허용...


In [ ]:
## 기존 CSV 정제만 수행
```
import pandas as pd
df0 = pd.read_csv('your_existing.csv')
from __main__ import sanitize_df
dfc = sanitize_df(df0)
dfc.to_csv('your_existing_clean.csv', index=False, encoding='utf-8-sig')
```